# 📖 Combined Example: Research Agent

---

---

## 🎯 Build a Complete Research Agent

By the end of this notebook, you'll build an agent that:
1. Takes a research query
2. Searches for information
3. Synthesizes findings
4. Returns a complete answer

In [ ]:
!pip install -q langgraph langchain-openai langchain-core
import os
if "OPENAI_API_KEY" not in os.environ:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI Key: ")

from langgraph.graph import StateGraph, END
from typing import TypedDict
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatOpenAI(model="gpt-4o-mini")

## 📦 Define State and Nodes

In [ ]:
class ResearchState(TypedDict):
    query: str
    search_results: list
    steps: list
    final_answer: str

# Node 1: Search
def search_node(state: ResearchState) -> ResearchState:
    query = state["query"]
    # Simulate search
    response = llm.invoke([HumanMessage(content=f"Search for: {query}")])
    state["search_results"].append(response.content)
    state["steps"].append(f"Searched for '{query}'")
    return state

# Node 2: Analyze
def analyze_node(state: ResearchState) -> ResearchState:
    query = state["query"]
    results = state.get("search_results", [])
    # Simulate analysis
    response = llm.invoke([HumanMessage(
        content=f"Given query '{query}' and results '{results}', provide an analysis"
    )])
    state["search_results"].append(response.content)
    state["steps"].append("Analyzed results")
    return state

# Node 3: Synthesize
def synthesize_node(state: ResearchState) -> ResearchState:
    query = state["query"]
    results = state.get("search_results", [])
    # Synthesize final answer
    response = llm.invoke([HumanMessage(
        content=f"Based on query '{query}', provide a comprehensive answer using this info: {results}"
    )])
    state["final_answer"] = response.content
    state["steps"].append("Synthesized final answer")
    return state

In [ ]:
# Build graph
graph = StateGraph(ResearchState)

graph.add_node("search", search_node)
graph.add_node("analyze", analyze_node)
graph.add_node("synthesize", synthesize_node)

graph.set_entry_point("search")
graph.add_edge("search", "analyze")
graph.add_edge("analyze", "synthesize")
graph.add_edge("synthesize", END)

app = graph.compile()

print("✅ Research Agent Built!")

In [ ]:
# Run research agent
initial_state = {
    "query": "What are the latest developments in AI agents?",
    "search_results": [],
    "steps": [],
    "final_answer": ""
}

result = app.invoke(initial_state)

print("📊 Research Complete!")
print("=" * 60)
print("Steps taken:")
for step in result["steps"]:
    print(f"  ✅ {step}")
print(f"\n📝 Final Answer:\n{result['final_answer'][:500]}...")

## 🧪 Try It Yourself
**Exercise**: Add a "fact-check" node to verify claims

## ✅ Summary

You built a complete research agent with:
1. Multiple processing stages
2. State passing between nodes
3. Synthesis for final output

## 🔗 Next
**[04_debugging_agents.ipynb](04_debugging_agents.ipynb)** - Debug common issues